In [1]:
import json


def calc_crc16(data: bytes) -> int:
    crc = 0xFFFF
    for byte in data:
        crc ^= byte
        for _ in range(8):
            crc = (crc >> 1) ^ 0xA001 if (crc & 1) else (crc >> 1)
    return crc & 0xFFFF


def voltage_to_percent(voltage: float) -> int:
    if voltage >= 4.2:
        return 100
    if voltage <= 2.7:
        return 0
    return int(round((voltage - 2.7) / (4.2 - 2.7) * 100))


def load_config(path="sensor_config.json") -> dict:
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        return {"field_mapping": {}}


def parse_wh65lp(raw: bytes) -> dict:
    """Декодирование данных метеостанции WH65LP (id=36) – 21 байт"""
    if len(raw) < 21:
        return {"error": "Too short"}
    d = raw
    dir_val = d[2] + ((d[3] & 0x80) << 1)
    tmp_val = d[4] + ((d[3] & 0x07) << 8)
    hum = d[5]
    wsp = d[6] + ((d[3] & 0x10) << 4) if (d[3] & 0x04) else d[6]
    gust = d[7]
    rain = (d[8] << 8) | d[9]
    uv = (d[10] << 8) | d[11]
    light = (d[12] << 16) | (d[13] << 8) | d[14]
    pres = (d[17] << 16) | (d[18] << 8) | d[19] if len(d) > 19 else 0

    return {
        "air_temperature": (tmp_val - 400) / 10.0 if tmp_val != 0x7FF else 0,
        "air_humidity": hum if hum != 0xFF else 0,
        "air_pressure": pres / 100.0 if pres != 0xFFFFFF else 0,
        "wind_direction": dir_val if dir_val != 0x1FF else 0,
        "wind_speed": (wsp / 8.0 * 0.51) if wsp != 0x1FF else 0,
        "wind_gust": gust * 0.51 if gust != 0xFF else 0,
        "rainfall": rain * 0.254,
        "uv": uv if uv != 0xFFFF else 0,
        "illumination": light / 10.0 if light != 0xFFFFFF else 0,
    }


def decode_sensor(data: bytes, sid: int, cfg: dict) -> dict:
    res = {"id_sens": sid, "raw": data.hex().upper(), "decoded": {}}

    if sid == 36:
        weather = parse_wh65lp(data)
        for name, value in weather.items():
            if isinstance(value, float):
                rounded = round(value, 1)
                res["decoded"][name] = {"value": rounded}
            else:
                res["decoded"][name] = {"value": value}
        return res

    fmap = cfg.get("field_mapping", {}).get(str(sid), [])
    if not fmap:
        res["decoded"] = {"raw": data.hex(), "note": "Нет конфигурации"}
        return res
    for f in fmap:
        off = (f["byte"] - 1) * 2
        if off + 2 <= len(data):
            raw_val = int.from_bytes(data[off : off + 2], "big")
            value = raw_val * f["coef"]
            rounded = round(value, 1)
            res["decoded"][f["name"]] = {
                "raw": raw_val,
                "coef": f["coef"],
                "value": rounded,
            }
    return res


def decode_hex_packet(
    hex_str: str, server_time: str = None, config_path: str = "sensor_config.json"
) -> str:
    config = load_config(config_path)
    hex_str = hex_str.replace(" ", "")
    data = bytes.fromhex(hex_str)
    off = 0

    # HEADER (16 байт)
    ccid = data[0:10].hex().upper()
    module_id = int.from_bytes(data[10:13], "big")
    bat_head_voltage = data[13] / 10.0
    signal = data[14]
    rssi = data[15]
    off = 16

    # FOOTER
    pkt_len = data[off]
    off += 1
    mod_id = int.from_bytes(data[off : off + 3], "big")
    off += 3
    bat_mem_voltage = data[off] / 10.0
    off += 1
    year = data[off]
    off += 1
    month = data[off]
    off += 1
    day = data[off]
    off += 1
    hour = data[off]
    off += 1
    minute = data[off]
    off += 1
    second = data[off]
    off += 1
    date_str = (
        f"20{year:02d}-{month:02d}-{day:02d} {hour:02d}:{minute:02d}:{second:02d}"
    )

    head_sens = {
        "ccid": ccid,
        "id_s": f"{module_id:08d}",
        "batt": voltage_to_percent(bat_head_voltage),
        "signal": signal,
    }
    if server_time:
        head_sens["server_received_at"] = server_time

    field_sens = {
        "id_s": f"{mod_id:08d}",
        "batt": voltage_to_percent(bat_mem_voltage),
        "time": date_str,
    }
    if rssi != 0:
        field_sens["signal"] = rssi

    result = {"head_sens": head_sens, "field_sens": field_sens, "ports": {}}

    while off < len(data) - 2:
        port = data[off]
        off += 1
        if port == 0:
            break
        address = data[off]
        op = data[off + 1]
        dlen = data[off + 2]
        off += 3
        sdata = data[off : off + dlen]
        off += dlen
        scrc = int.from_bytes(data[off : off + 2], "little")
        off += 2

        decoded = decode_sensor(sdata, address, config)

        sensor_item = {"address": address}
        for name, val in decoded.get("decoded", {}).items():
            if isinstance(val, dict) and "value" in val:
                sensor_item[name] = val["value"]
            else:
                sensor_item[name] = val

        port_key = f"port_{port}"
        result["ports"].setdefault(port_key, []).append(sensor_item)

    total_block = data[16:off]
    total_crc = int.from_bytes(data[off : off + 2], "little")
    result["crc_total"] = total_crc
    result["crc_total_valid"] = calc_crc16(total_block) == total_crc

    return json.dumps(result, indent=2, ensure_ascii=False)

In [2]:
hex_input = ( "89701018292556469187004E222843003F004E22281A050F1227300000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000D9B1")
json_str = decode_hex_packet(hex_input, server_time="2026-05-14 12:00:00")
print(json_str)

{
  "head_sens": {
    "ccid": "89701018292556469187",
    "id_s": "00020002",
    "batt": 87,
    "signal": 67,
    "server_received_at": "2026-05-14 12:00:00"
  },
  "field_sens": {
    "id_s": "00020002",
    "batt": 87,
    "time": "2026-05-15 18:39:48"
  },
  "ports": {},
  "crc_total": 0,
  "crc_total_valid": false
}
